In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
init_load_flag = int(dbutils.widgets.get("init_load_flag"))

### **Data Reading From Source**

In [0]:
df = spark.sql('''select * from retail.silver.customers_silver''')

In [0]:
df.display()

**Removing Duplicates**

In [0]:
df = df.dropDuplicates(subset=['customer_id'])

In [0]:
df.display()

## **Differentiating new and old records**

In [0]:
if init_load_flag == 0:

    df_old = spark.sql('''select DimCustomerKey, customer_id, create_date, update_date
                        from retail.gold.DimCustomers''')
else:
    df_old = spark.sql('''select 0 DimCustomerKey, 0 customer_id, 0 create_date, 0 update_date
                        from retail.silver.customers_silver where 1=0''') 

In [0]:
df_old = df_old.withColumnRenamed("DimCustomerKey", "old_DimCustomerKey")\
        .withColumnRenamed("customer_id", "old_customer_id")\
        .withColumnRenamed("update_date", "old_update_date")\
        .withColumnRenamed("create_date", "old_create_date")



In [0]:
df_old.display()

**Applying Join with the old Records**

In [0]:
df_join = df.join(df_old, df.customer_id == df_old.old_customer_id, 'left')
df_join.display()

In [0]:
df_new = df_join.filter(df_join.old_DimCustomerKey.isNull())
df_new.display()

In [0]:
df_old = df_join.filter(df_join.old_DimCustomerKey.isNotNull())
df_old.display()

**Preparing df_old**

In [0]:
# Dropping all the columns which are not required
df_old = df_old.drop('old_customer_id','old_update_date')

# Renaming "old_create_date" & "old_DimCustomerKey" column to match the target 
df_old = df_old.withColumnRenamed('old_create_date', 'create_date')
df_old = df_old.withColumnRenamed('old_DimCustomerKey', 'DimCustomerKey')
df_old = df_old.withColumn('create_date',to_timestamp(col('create_date')))

# adding update_date with current timestamp

df_old = df_old.withColumn("update_date", current_timestamp())

In [0]:
df_old.display()

**Preparing df_new**

In [0]:
# Dropping all the columns which are not required
df_new = df_new.drop('old_DimCustomerKey','old_customer_id','old_update_date','old_create_date')


# Recreating update_date, current_date with current timestamp

df_new = df_new.withColumn("update_date", current_timestamp())
df_new = df_new.withColumn("create_date", current_timestamp())

In [0]:
df_new.display()

**Surrogate Key - From 1**


In [0]:
df_new = df_new.withColumn("DimCustomerKey",monotonically_increasing_id()+lit(1))

**Adding Max Surrogate Key so the new rows get ids from there on**

In [0]:
if init_load_flag == 1:
    max_surrogate_key = 0
else:
    df_maxsur = spark.sql('''select max(DimCustomerKey) as max_surrogate_key from retail.gold.DimCustomers''')
    max_surrogate_key = df_maxsur.collect()[0]['max_surrogate_key']



In [0]:
df_new = df_new.withColumn("DimCustomerKey", lit(max_surrogate_key)+col("DimCustomerKey"))

In [0]:
df_new.display()

**Union of df_new and df_old**

In [0]:
df_final = df_new.unionByName(df_old)




In [0]:
df_final.display()

## **SCD Type - 1**

In [0]:
from delta.tables import DeltaTable

In [0]:
if init_load_flag == 1:
    df_final.write.mode("overwrite")\
        .format('delta')\
        .option("path","abfss://gold@stretaildevwestindia.dfs.core.windows.net/DimCustomers")\
        .saveAsTable("retail.gold.DimCustomers")
else:
    dlt_obj = DeltaTable.forPath(spark, "abfss://gold@stretaildevwestindia.dfs.core.windows.net/DimCustomers") #target

    dlt_obj.alias("tgt").merge(df_final.alias("src"), "tgt.DimCustomerKey = src.DimCustomerKey")\
        .whenMatchedUpdateAll()\
            .whenNotMatchedInsertAll()\
                .execute()



In [0]:
%sql
select * from retail.gold.dimcustomers